# VELNOR Qwen3-8B QLoRA - continuation run

Resumes the adapter from a previous real run and trains 3 more epochs on the same curriculum. The prior best validation loss is carried forward, so the continuation only saves an adapter when it actually beats 1.19551.

In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError('No CUDA GPU attached. Select a free GPU runtime; CPU training is disabled.')
p = torch.cuda.get_device_properties(0)
free, total = torch.cuda.mem_get_info(0)
print('GPU:', p.name, '| total GiB:', round(p.total_memory/1024**3,2), '| free GiB:', round(free/1024**3,2))
if p.total_memory < 14*1024**3 or free < 12*1024**3:
    raise RuntimeError('GPU does not meet Qwen3-8B QLoRA preflight. No training started.')

In [ ]:
import subprocess
REPO='/content/velnor-qwen3-8b'
subprocess.run(['rm','-rf',REPO])
subprocess.run(['git','clone','--branch','feat/qwen3-8b-qlora','--depth','1','https://github.com/vectolith/velnor.git',REPO], check=True)
print(subprocess.check_output(['git','-C',REPO,'rev-parse','HEAD'], text=True).strip())

In [ ]:
%cd /content/velnor-qwen3-8b
!python -m pip install -q -r training/requirements-qwen3-qlora.txt
import torch
assert torch.cuda.is_available()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import os, subprocess
PRIOR = Path('/content/drive/MyDrive/VELNOR-training-runs/qwen3-8b-20261112T123354Z')
assert (PRIOR/'outputs'/'adapter'/'adapter_config.json').is_file(), 'prior adapter missing'
run_name = 'qwen3-8b-cont-' + subprocess.check_output(['date','-u','+%Y%m%dT%H%M%SZ'], text=True).strip()
RUN_DIR = Path('/content/drive/MyDrive/VELNOR-training-runs') / run_name
OUT_DIR = RUN_DIR / 'outputs'
OUT_DIR.mkdir(parents=True, exist_ok=False)
os.environ['VELNOR_QWEN3_OUTPUT'] = str(OUT_DIR)
os.environ['VELNOR_QWEN3_RESUME'] = str(PRIOR/'outputs'/'adapter')
os.environ['VELNOR_QWEN3_PREV_BEST_LOSS'] = '1.19551'
os.environ['VELNOR_QWEN3_MAX_SEQ_LEN'] = '256'
os.environ['VELNOR_QWEN3_EPOCHS'] = '3'
os.environ['VELNOR_QWEN3_GRAD_ACCUM'] = '8'
print('run dir:', RUN_DIR)

In [ ]:
%%bash
set -euo pipefail
cd /content/velnor-qwen3-8b/training
python -u train_qwen3_8b_qlora.py 2>&1 | tee "$VELNOR_QWEN3_OUTPUT/training.log"
test -s "$VELNOR_QWEN3_OUTPUT/adapter/adapter_config.json"
test -s "$VELNOR_QWEN3_OUTPUT/training_report.json"
echo DONE